# Therapy prediction plots: 3D plots of a, f, and d

This file generates Figure 7 in the main text, and Supplementary Figure S3.

It requires that the parameter estimation has been completed and there exists a file with those results. In our case we load a subset of the full result set because of the size of the result space. Edit the location of those files below. It loads the point cloud from that file and, for a selected initial ratio and value of $m$, it turns the shapes of the proliferative subclone, invasive subclone, and tumor elimination area (as appropriate) into 3D ($a$, $f$, $d$)-space meshes using PyVista. These meshes are plotted in 3D space and saved if desired.

Input group, $m$ value, and $d$ values in cell 10.

## Prep

Load needed packages

In [ ]:
%matplotlib widget
import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import math
import seaborn as sns
import os
import time
from sklearn.linear_model import LinearRegression
from estimator import Estimator
from statsmodels.formula import api as smf
import pyvista as pv
pv.start_xvfb()
pv.set_jupyter_backend("static")

Close any figures generated from previous runs

In [ ]:
plt.close("all")

## Define data and user input-parameters

Define data files, groups to plot, and save path.
`result_subset_file` should be a file containing the set (or subset) or acceptable parameter combinations as determined by the script for Figure 5b (if you include a save_file for script 5b, it will save out this data).

In [ ]:
result_subset_file = "result_file.csv"
groups = ["Grp. A1 B6 (100% C1)", "Grp. A2 B6 (80% C1; 20% C11)", "Grp. A3 B6 (50% C1; 50% C11)", "Grp. A4 B6 (20% C1; 80% C11)", "Grp. A5 B6 (100% C11)"]
save_path = "figures/daf/" # Set to None to not save the figures

In [ ]:
result_subset = pd.read_csv(result_subset_file)
result_subset

## Functions

Function to find the next clockwise point.

@param p : current point [<i>x</i>, <i>y</i>]

@param b : last point [<i>x</i>, <i>y</i>]

@param max_x : the maximum possible <i>x</i> value

@param max_y : the maximum possible <i>y</i> value

In [ ]:
def next_clockwise(p, b, max_x, max_y):
    # Out is next clockwise point
    out = None
    
    # Compare current point to previous point to find the next clockwise point
    if b[0] < p[0]: # we're currently to the left
        if b[1] < p[1]: # we're currently above
            out = [b[0]+1, b[1]] # move right
        else: # we're currently equal y or below
            out = [b[0], b[1]-1] # move up
    elif b[0] == p[0]: # we're currently same x
        if b[1] < p[1]: # we're currently above
            out = [b[0]+1, b[1]] # move right
        else: # we're currently below
            out = [b[0]-1, b[1]] # move left
    else: # b[0] > p[0], we're currently to the right
        if b[1] <= p[1]: # we're currently above or equal y
            out = [b[0], b[1]+1] # move down
        else: # we're currently below
            out = [b[0]-1, b[1]] # move left
    
    # If the next clockwise point is out of bounds, call function again setting the previous point to out of bounds next point
    if out[0] < 0 or out[1] < 0: return next_clockwise(p, out, max_x, max_y)
    if out[0] > max_x or out[1] > max_y: return next_clockwise(p, out, max_x, max_y)
    
    # Return next clockwise point
    return out

Moore-neighbor tracing algorithm: 
https://en.wikipedia.org/wiki/Moore_neighborhood#Algorithm

@param df : 2D dataframe with 0s where C1 (proliferative) wins, 1 where C11 (invasive) wins, and 2s where both subclones are eliminated

@param v : Binary flag for which subclone we want to trace. 0 for C1 (proliferative), 1 for C11 (invasive)

@return : Returns a list of points that outline the shape defined by the points in the dataframe associated with the given subclone

In [ ]:
def mnt(df, v):
    # trace will keep track of the points we've visited in order
    trace = []
    
    # Find a starting point for the given subclone, looping through the grid from left to right, top to bottom
    start_loc = None
    for y in range(len(df.index)):
        for x in range(len(df.columns)):
            if df.iloc[y,x] == v:
                start_loc = [x,y]
                break
        # Exit the loop if we've found a starting value
        if start_loc != None: break
    # If there are no values of the given color, exit
    if start_loc == None: return trace
        
    # Append our starting location to our trace
    trace.append([df.columns[start_loc[0]], df.index[start_loc[1]]])

    # Set maximum sizes as the sizes of the grid
    max_x = len(df.columns)-1
    max_y = len(df.index)-1

    # p is our current location, set to starting point
    p = start_loc
    # b is our previous point, set it one to the left
    b = [start_loc[0]-1, start_loc[1]]
    # c is the next clockwise point around p starting from b
    c = next_clockwise(p, b, max_x, max_y)

    # Loop until we are back to the starting point
    while c[0] != start_loc[0] or c[1] != start_loc[1]:
        # If our next clockwise point is the correct color:
        if df.iloc[c[1], c[0]] == v:
            trace.append([df.columns[c[0]], df.index[c[1]]]) # Add it to the trace
            b = p # Last point is now first point
            p = c # Current point is now next point
            c = next_clockwise(p, b, max_x, max_y) # Find next point
        # If the next clockwise point is not the correct color
        else:
            b = c # Our last point is the last clockwise point
            c = next_clockwise(p, b, max_x, max_y) # Find next clockwise point
    
    # Add the first point to the end of the trace
    trace.append(trace[0])
    # Return the trace
    return trace

Turn the list of points returned from the mnt function into a set of lines.

@param points : list of points defining the outline of the shape (returned from mnt())

In [ ]:
def get_lines(points):
    # List of lines to return
    # Format is [[slope1, intercept1, min_value, max_value], ...]
    lines = []

    # Loop through all the points
    for i in range(len(points)-1):
        # If the current point and next point form a vertical line
        if points[i][0] == points[i+1][0]:
            slope = np.inf # Set the slope to infinity
            intercept = points[i][0] # Set the intercept to the x value
            # If the slope and intercept are the same as the previous line
            if lines != [] and slope == lines[-1][0] and intercept == lines[-1][1]:
                # Just update the min and max of the previous line
                lines[-1][2] = min(lines[-1][2], points[i][1], points[i+1][1])
                lines[-1][3] = max(lines[-1][3], points[i][1], points[i+1][1])
            # If the slope and/or intercept are different, add the new line to the list
            else:
                lines.append([slope, intercept, min(points[i][1], points[i+1][1]), max(points[i][1], points[i+1][1])])
        # If the two points don't form a vertical line
        else:
            # Calculate the slope and intercept
            slope = round((points[i][1] - points[i+1][1]) / (points[i][0] - points[i+1][0]), 5)
            intercept = round(points[i][1] - slope * points[i][0], 5)
            # If the slope and intercept are the same as the previous line
            if lines != [] and slope == lines[-1][0] and intercept == lines[-1][1]:
                # Just update min and max of the previous line
                lines[-1][2] = min(lines[-1][2], points[i][0], points[i+1][0])
                lines[-1][3] = max(lines[-1][3], points[i][0], points[i+1][0])
            # If the slope and/or intercept are different, add the new line to the list
            else:
                lines.append([slope, intercept, min(points[i][0], points[i+1][0]), max(points[i][0], points[i+1][0])])
    # Return the list of line data
    return lines

Plot the lines output from get_lines

@param ax : The axis on which to plot

@param lines : A list of line information in the format [[slope1, intercept1, minx1, maxx1], [slope2, intercept2, minx2, maxx2], ...]

@param color : The color to plot the lines in. Must be an appropriate Matplotlib color: https://matplotlib.org/stable/users/explain/colors/colors.html#colors-def

In [ ]:
def plotlines(ax, lines, color="black"):
    for line in lines:
        if line[0] == np.inf: # vertical line
            ax.vlines(line[1], line[2], line[3], color=color)
        ax.plot(np.arange(line[2], line[3]+0.01, 0.01), [line[0]*x+line[1] for x in np.arange(line[2], line[3]+0.01, 0.01)], color=color)

Function to use PyVista to plot 3D shapes.

@param ds : List of <i>d</i> values of the <i>z</i> slices to plot

@param dfs : List of 2D data frames, each representing the slice of the 3D space associated with each d value

@param curr_group : The mouse group we're plotting (for save file)

@param curr_m : The <i>m</i> value for the current 3D data frame (for save file)

@param stretch : The amount to shift the <i>a</i> and <i>f</i> values of the blocks in order to more clearly see the boundaries of the shapes

In [ ]:
def plot_3d(ds, dfs, curr_group, curr_m, stretch=0):
    # Create a list of d values, corresponding to each true d value, that match the range of a and f
    fake_ds = np.linspace(0, 2.5, len(ds))

    # Initiate plotter
    pl = pv.Plotter(notebook=True)

    # C1 (proliferative)
    # Get traces for each dataframe (d-slice)
    traces = [mnt(dfx, 0) for dfx in dfs]
    # Update a and f values with stretch and switch the d values to the scaled d values
    traces2 = []
    for i in range(len(ds)):
        t = []
        for af in traces[i]:
            t += [[af[0]+stretch, af[1]-stretch, fake_ds[i]]]
        traces2 += t
    # Only plot if there are points where C1 (proliferative) wins
    if len(traces2) > 0:
        # Use PyVista to plot the 3D shape
        cloud = pv.PolyData(traces2)
        volume = cloud.delaunay_3d(alpha=0)
        shell = volume.extract_geometry()
        if shell.n_points > 0:
            pl.add_mesh(shell, color="#ffa500", opacity=0.5)

    # C11 (invasive)
    # Get traces for each dataframe (d-slice)
    traces = [mnt(dfx, 1) for dfx in dfs]
    # Update a and f values with stretch and switch the d values to the scaled d values
    traces2 = []
    for i in range(len(ds)):
        t = []
        for af in traces[i]:
            t += [[af[0]-stretch, af[1]+stretch, fake_ds[i]]]
        traces2 += t
    # Only plot if there are points where C11 (invasive) wins
    if len(traces2) > 0:
        # Use PyVista to plot the 3D shape
        cloud = pv.PolyData(traces2)
        volume = cloud.delaunay_3d(alpha=0)
        shell = volume.extract_geometry()
        if shell.n_points > 0:
            pl.add_mesh(shell, color="#0000ff", opacity=0.4)

    # Tumor elimination
    # Get traces for each dataframe (d-slice)
    traces = [mnt(dfx, 2) for dfx in dfs]
    # Switch the d values to the scaled d values (don't move the elimination shape with shift)
    traces2 = []
    for i in range(len(ds)):
        t = []
        for af in traces[i]:
            t += [[af[0], af[1], fake_ds[i]]]
        traces2 += t
    # Only plot if there are points where both subclones are eliminated
    if len(traces2) > 0:
        # Use PyVista to plot the 3D shape
        cloud = pv.PolyData(traces2)
        volume = cloud.delaunay_3d(alpha=0)
        shell = volume.extract_geometry()
        if shell.n_points > 0:
            pl.add_mesh(shell, color="green", opacity=0.5)

    # Set bounds and grid
    pl.show_bounds(location="outer", grid="front")
    # Set the camera settings
    pl.camera_position = 'yz'
    pl.camera.elevation=30
    pl.camera.azimuth = 52
    # Label the axis
    labels = dict(xtitle="f", ytitle="a", ztitle="d")
    pl.show_grid(**labels, font_size=20, font_family="arial", bold=False, n_xlabels=6, n_ylabels=6, n_zlabels=5)

    # Save figure
    if save_path:
        pl.save_graphic("{}{}_m{}.svg".format(save_path, groups[curr_group][5:7], curr_m))

    # Show figure
    pl.show()

## Set parameters values to use

In [ ]:
# Index of the group to plot
curr_group = 0
# m value for the data to plot
curr_m = -0.1
# d values of the slices to use to compute the shape
ds = [-2, -4, -6, -8, -10]

## Extract and format data

In [ ]:
temp = result_subset[(result_subset["group"] == groups[curr_group]) & (result_subset["m"] == curr_m)]
temp = temp[["d", "a", "f", "winner"]]
temp = temp.drop_duplicates()
temp = temp.groupby(["d", "a", "f"]).max().reset_index()

# Dataframes associated with d values
dfs = [pd.pivot(temp[temp["d"] == d], index="a", columns="f", values="winner") for d in ds]

## Plot figure

Note that the plotted figure will have the scaled <i>d</i> values, not the true <i>d</i> values.

In [ ]:
plot_3d(ds, dfs, curr_group, curr_m, stretch=0)